# Chapter 14 — End-to-End Pipeline Evaluation (Colab)

Standalone notebook that runs the **complete** PPE-compliance pipeline on the held-out test images and scores it as a single system.

```
image  ->  Stage 1: YOLOv8n          ->  Person bounding boxes
       ->  Stage 2: ResNet50 (FT)    ->  per-crop helmet / vest probabilities
       ->  threshold @ 0.5           ->  booleans
       ->  Stage 3: compliance()     ->  per-worker verdict (green / red)
```

**Why a separate Colab notebook?** The integration needs *both* backends at once: YOLO runs on PyTorch and the classifier on TensorFlow. TensorFlow has no GPU on native Windows (≥2.11) and the laptop runs Python 3.14 (no TF wheels at all), so the only place that has everything with a GPU is Colab. The polished deliverable `01_main_pipeline.ipynb` (Chapter 14) only *displays* the artifacts produced here.

**What it measures.** For every test image we run the detector, classify each detected person, and produce a compliance verdict. We then match the predicted person boxes to the ground-truth person boxes (greedy IoU ≥ 0.5) and, on the matched workers, ask: *did the full pipeline get the compliance verdict right?* We report detection recall alongside, so the Stage 1 + Stage 2 errors are visible as layered.

**Two test splits** (the same Cross-Dataset split used throughout the project):
- **Test In-Domain** — Kaggle CSS test images.
- **Test Out-of-Domain** — the entire Ultralytics Construction-PPE dataset (a domain never seen in training).


## Before you run

1. **Fine-tuned model in Drive.** `MyDrive/PPE_Project/stage2_results/resnet50_finetuned/best.keras` must exist (produced by `stage2_colab.ipynb`).
2. **Datasets in Drive.** `MyDrive/Deep_Learning_course/datasets/` must contain `construction-ppe/` and `kaggle-css/`.
3. **GPU runtime.** `Runtime → Change runtime type → T4 GPU` (or L4).
4. **Run all cells** (`Runtime → Run all`). Evaluation is batched (YOLO + classifier run on chunks of images), so the out-of-domain split — the entire Ultralytics dataset — finishes in a few minutes. A `tqdm` progress bar shows live progress per split.


In [ ]:
# Mount Google Drive and resolve all paths
from google.colab import drive
drive.mount("/content/drive")

import os

DRIVE_ROOT   = "/content/drive/MyDrive/PPE_Project"
DATA_ROOT    = "/content/drive/MyDrive/Deep_Learning_course/datasets"
FT_MODEL     = f"{DRIVE_ROOT}/stage2_results/resnet50_finetuned/best.keras"
OUT_DIR      = f"{DRIVE_ROOT}/end_to_end"
os.makedirs(OUT_DIR, exist_ok=True)

# Dataset sub-roots (must match Chapter 3 of the main notebook)
ULTRALYTICS_DIR = f"{DATA_ROOT}/construction-ppe"
KAGGLE_CSS_DIR  = f"{DATA_ROOT}/kaggle-css/css-data"

assert os.path.exists(FT_MODEL),        f"Missing FT model: {FT_MODEL}"
assert os.path.exists(ULTRALYTICS_DIR), f"Missing Ultralytics: {ULTRALYTICS_DIR}"
assert os.path.exists(KAGGLE_CSS_DIR),  f"Missing Kaggle CSS: {KAGGLE_CSS_DIR}"

print("FT model     :", FT_MODEL)
print("Ultralytics  :", ULTRALYTICS_DIR)
print("Kaggle CSS   :", KAGGLE_CSS_DIR)
print("Output dir   :", OUT_DIR)


In [ ]:
# Clone the repo to obtain the trained YOLO weights (best.pt lives in git)
REPO = "/content/ppe-compliance-detection"
if not os.path.exists(REPO):
    os.system(f"git clone -q https://github.com/Razelbaz1/ppe-compliance-detection.git {REPO}")

YOLO_WEIGHTS = f"{REPO}/results/stage1/yolov8n_50e/weights/best.pt"
assert os.path.exists(YOLO_WEIGHTS), f"Missing YOLO weights: {YOLO_WEIGHTS}"
print("YOLO weights :", YOLO_WEIGHTS)


In [ ]:
# Install ultralytics (PyTorch is already present on Colab) and import everything
os.system("pip -q install ultralytics")

import glob
import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow import keras
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- Pipeline configuration --------------------------------------------------
IMG_SIZE   = 224     # ResNet50 input side (must match Stage 2 training)
CONF_THR   = 0.25    # YOLO confidence operating point
PPE_THR    = 0.50    # helmet/vest probability -> boolean threshold
MATCH_IOU  = 0.50    # IoU to match a predicted box to a ground-truth person

# --- Load both models --------------------------------------------------------
print("Loading detector (YOLOv8n) ...")
detector = YOLO(YOLO_WEIGHTS)

print("Loading classifier (ResNet50 fine-tuned) ...")
classifier = keras.models.load_model(FT_MODEL)

print(f"TF {tf.__version__} | GPU: {bool(tf.config.list_physical_devices('GPU'))}")
print("Models loaded.")


In [ ]:
# =============================================================================
# Geometry + ground-truth helpers (identical rule to Chapter 10, "Strategy A")
# =============================================================================
def yolo_to_pixel(cx, cy, w, h, img_w, img_h):
    """YOLO normalized (cx, cy, w, h) -> pixel (x1, y1, x2, y2)."""
    xc, yc = cx * img_w, cy * img_h
    bw, bh = w * img_w, h * img_h
    x1 = max(0, int(round(xc - bw / 2)))
    y1 = max(0, int(round(yc - bh / 2)))
    x2 = min(img_w, int(round(xc + bw / 2)))
    y2 = min(img_h, int(round(yc + bh / 2)))
    return x1, y1, x2, y2


def compute_iou(a, b):
    """IoU between two (x1, y1, x2, y2) boxes."""
    xa, ya = max(a[0], b[0]), max(a[1], b[1])
    xb, yb = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xb - xa) * max(0, yb - ya)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def is_worn_by(person_box, item_box, iou_thr=0.10):
    """True if item_box belongs to the person (overlap or center inside)."""
    if compute_iou(person_box, item_box) > iou_thr:
        return True
    cx = (item_box[0] + item_box[2]) / 2
    cy = (item_box[1] + item_box[3]) / 2
    return (person_box[0] <= cx <= person_box[2] and
            person_box[1] <= cy <= person_box[3])


def parse_ground_truth(label_path, img_w, img_h, person_cls, helmet_cls, vest_cls):
    """Read one YOLO label file -> list of (person_box, gt_helmet, gt_vest).

    gt_helmet/gt_vest are derived with the same Strategy-A rule used to build
    the Stage 2 crop dataset: a person 'has' an item if any item bbox is worn
    by them (IoU > 0.1 or item center inside the person box).
    """
    if not os.path.exists(label_path):
        return []
    person_boxes, helmet_boxes, vest_boxes = [], [], []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cid = int(parts[0])
            box = yolo_to_pixel(*map(float, parts[1:]), img_w, img_h)
            if cid == person_cls:   person_boxes.append(box)
            elif cid == helmet_cls: helmet_boxes.append(box)
            elif cid == vest_cls:   vest_boxes.append(box)

    out = []
    for p_box in person_boxes:
        gt_helmet = int(any(is_worn_by(p_box, h) for h in helmet_boxes))
        gt_vest   = int(any(is_worn_by(p_box, v) for v in vest_boxes))
        out.append((p_box, gt_helmet, gt_vest))
    return out


# --- Stage 3 (identical to Chapter 13 of the main notebook) ------------------
REQUIRED_PPE = ("helmet", "vest")

def compliance(helmet: bool, vest: bool) -> dict:
    present = {"helmet": bool(helmet), "vest": bool(vest)}
    missing = [item for item in REQUIRED_PPE if not present[item]]
    return {"compliant": len(missing) == 0, "missing_items": missing}

print("Helpers and Stage 3 logic defined.")


In [ ]:
# =============================================================================
# The end-to-end pipeline (Stage 1 -> Stage 2 -> Stage 3)
# =============================================================================
def classify(crops_rgb):
    """List of HxWx3 uint8 RGB crops -> (N, 2) array of [helmet, vest] probs.

    Uses a DIRECT model call -- classifier(batch, training=False) -- instead of
    classifier.predict(). For many small batches in a loop the direct call has a
    fraction of the overhead and is numerically identical (same forward pass, in
    inference mode). The Stage 2 model applies preprocess_input internally, so we
    feed raw RGB in [0, 255] exactly as during training (crops read as RGB).
    """
    if len(crops_rgb) == 0:
        return np.zeros((0, 2), dtype="float32")
    batch = np.stack([cv2.resize(c, (IMG_SIZE, IMG_SIZE)) for c in crops_rgb]).astype("float32")
    return classifier(batch, training=False).numpy()


def yolo_boxes(res):
    """One ultralytics Result -> (boxes Nx4 int xyxy, confs N)."""
    if res.boxes is None or len(res.boxes) == 0:
        return np.zeros((0, 4), dtype=int), np.zeros((0,), dtype="float32")
    return res.boxes.xyxy.cpu().numpy().astype(int), res.boxes.conf.cpu().numpy()


def detect_and_crop(img_bgr, res):
    """One image + its YOLO result -> (crops_rgb, boxes_clipped, confs).

    Clips each box to the image, drops degenerate ones, and converts the crop
    BGR -> RGB (the classifier was trained on RGB crops).
    """
    H, W = img_bgr.shape[:2]
    raw_boxes, raw_confs = yolo_boxes(res)
    crops, boxes, confs = [], [], []
    for (x1, y1, x2, y2), cf in zip(raw_boxes, raw_confs):
        x1, y1 = max(0, int(x1)), max(0, int(y1))
        x2, y2 = min(W, int(x2)), min(H, int(y2))
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue
        crops.append(cv2.cvtColor(img_bgr[y1:y2, x1:x2], cv2.COLOR_BGR2RGB))
        boxes.append((x1, y1, x2, y2))
        confs.append(float(cf))
    return crops, boxes, confs


def workers_from(boxes, confs, probs):
    """Combine detections + classifier probs into worker verdict dicts."""
    workers = []
    for box, cf, pr in zip(boxes, confs, probs):
        helmet = bool(pr[0] >= PPE_THR)
        vest   = bool(pr[1] >= PPE_THR)
        verdict = compliance(helmet, vest)
        workers.append({
            "box": box, "conf": cf, "helmet": helmet, "vest": vest,
            "compliant": verdict["compliant"], "missing": verdict["missing_items"],
        })
    return workers


def run_pipeline(img_bgr):
    """Full pipeline on a single BGR image -> list of worker dicts.

    Convenience wrapper for one image (used by the qualitative sample grid).
    The bulk evaluation in evaluate_split() batches across images for speed.
    """
    res = detector.predict(img_bgr, conf=CONF_THR, verbose=False)[0]
    crops, boxes, confs = detect_and_crop(img_bgr, res)
    return workers_from(boxes, confs, classify(crops))

print("Pipeline functions defined (classify uses a direct model call).")


In [ ]:
# =============================================================================
# Evaluate the pipeline over a whole split (batched + greedy GT matching)
# =============================================================================
def list_pairs(images_dir, labels_dir):
    """Return [(image_path, label_path), ...] for every image with a label."""
    pairs = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        for img_path in sorted(glob.glob(os.path.join(images_dir, ext))):
            stem = os.path.splitext(os.path.basename(img_path))[0]
            pairs.append((img_path, os.path.join(labels_dir, stem + ".txt")))
    return pairs


def _match_and_score(workers, gts, acc):
    """Greedy-match predicted workers to GT persons; update the accumulators.

    Highest-confidence predictions are matched first to the unused GT person
    with IoU >= MATCH_IOU. Compliance / helmet / vest correctness is tallied on
    the matched pairs. acc["cm"][g, p] counts (GT compliant=g, predicted=p).
    """
    acc["n_gt"]   += len(gts)
    acc["n_pred"] += len(workers)
    used = set()
    for i in sorted(range(len(workers)), key=lambda i: -workers[i]["conf"]):
        pb = workers[i]["box"]
        best_j, best_iou = -1, MATCH_IOU
        for j, (gb, gh, gv) in enumerate(gts):
            if j in used:
                continue
            iou = compute_iou(pb, gb)
            if iou >= best_iou:
                best_iou, best_j = iou, j
        if best_j < 0:
            continue
        used.add(best_j)
        _, gh, gv = gts[best_j]
        g_comp = int(gh == 1 and gv == 1)
        p_comp = int(workers[i]["compliant"])
        acc["n_matched"] += 1
        acc["comp_ok"]   += int(p_comp == g_comp)
        acc["helmet_ok"] += int(workers[i]["helmet"] == bool(gh))
        acc["vest_ok"]   += int(workers[i]["vest"]   == bool(gv))
        acc["cm"][g_comp, p_comp] += 1


def evaluate_split(pairs, person_cls, helmet_cls, vest_cls, img_batch=16):
    """Run the pipeline over a split and accumulate metrics.

    SPEED: images are processed in chunks of `img_batch`. YOLO runs on the whole
    chunk at once, and the classifier runs ONE batch over every crop collected
    from the chunk -- so the GPU receives large batches instead of one image at a
    time. The numbers are identical to a per-image loop; only the batching (and
    therefore the speed) differs.

    SCORING: predicted boxes are greedily matched to GT persons (see
    _match_and_score). Compliance / helmet / vest accuracy are measured on the
    matched workers; detection recall + precision quantify Stage 1 separately.
    Returns (metrics_dict, confusion_matrix) with cm[g, p] = #(GT comp=g, pred=p).
    """
    acc = {"n_gt": 0, "n_pred": 0, "n_matched": 0,
           "comp_ok": 0, "helmet_ok": 0, "vest_ok": 0,
           "cm": np.zeros((2, 2), dtype=int)}

    for start in tqdm(range(0, len(pairs), img_batch), desc="batches", unit="batch"):
        chunk = pairs[start:start + img_batch]

        # Load images, keep only those that decode
        loaded = [(cv2.imread(p), lbl) for p, lbl in chunk]
        loaded = [(im, lbl) for im, lbl in loaded if im is not None]
        if not loaded:
            continue

        # Stage 1: detect on the whole chunk at once
        results = detector.predict([im for im, _ in loaded], conf=CONF_THR, verbose=False)

        # Collect crops across the chunk, remembering how many belong to each image
        per_image, all_crops = [], []
        for (im, _), res in zip(loaded, results):
            crops, boxes, confs = detect_and_crop(im, res)
            per_image.append((boxes, confs, len(crops)))
            all_crops.extend(crops)

        # Stage 2: classify every crop of the chunk in ONE call
        probs = classify(all_crops)

        # Stage 3 + scoring, per image (slice this image's probs back out)
        offset = 0
        for (im, lbl_path), (boxes, confs, n_crops) in zip(loaded, per_image):
            workers = workers_from(boxes, confs, probs[offset:offset + n_crops])
            offset += n_crops
            H, W = im.shape[:2]
            gts = parse_ground_truth(lbl_path, W, H, person_cls, helmet_cls, vest_cls)
            _match_and_score(workers, gts, acc)

    m = max(1, acc["n_matched"])
    metrics = {
        "Images":          len(pairs),
        "GT persons":      acc["n_gt"],
        "Pred persons":    acc["n_pred"],
        "Matched":         acc["n_matched"],
        "Det recall":      acc["n_matched"] / max(1, acc["n_gt"]),
        "Det precision":   acc["n_matched"] / max(1, acc["n_pred"]),
        "Compliance acc":  acc["comp_ok"] / m,
        "Helmet acc":      acc["helmet_ok"] / m,
        "Vest acc":        acc["vest_ok"] / m,
    }
    return metrics, acc["cm"]

print("evaluate_split() defined (batched).")


In [ ]:
# =============================================================================
# Run on both test splits
# =============================================================================
# Kaggle CSS class ids:  Person=5, Hardhat=0, Safety Vest=7
# Ultralytics class ids: person=6, helmet=0, vest=2

# In-Domain = Kaggle CSS test split
in_domain_pairs = list_pairs(
    f"{KAGGLE_CSS_DIR}/test/images",
    f"{KAGGLE_CSS_DIR}/test/labels",
)

# Out-of-Domain = the ENTIRE Ultralytics dataset (train + val + test)
ood_pairs = []
for s in ("train", "val", "test"):
    ood_pairs += list_pairs(
        f"{ULTRALYTICS_DIR}/images/{s}",
        f"{ULTRALYTICS_DIR}/labels/{s}",
    )

print(f"In-Domain images     : {len(in_domain_pairs):,}")
print(f"Out-of-Domain images : {len(ood_pairs):,}")

print("\nEvaluating Test In-Domain (Kaggle CSS test) ...")
in_metrics, in_cm = evaluate_split(in_domain_pairs, person_cls=5, helmet_cls=0, vest_cls=7)

print("Evaluating Test Out-of-Domain (Ultralytics) ...")
ood_metrics, ood_cm = evaluate_split(ood_pairs, person_cls=6, helmet_cls=0, vest_cls=2)

summary = pd.DataFrame([
    {"Split": "Test In-Domain (Kaggle test)",       **in_metrics},
    {"Split": "Test Out-of-Domain (Ultralytics)",   **ood_metrics},
])

fmt = {
    "Images":         lambda v: f"{int(v):,}",
    "GT persons":     lambda v: f"{int(v):,}",
    "Pred persons":   lambda v: f"{int(v):,}",
    "Matched":        lambda v: f"{int(v):,}",
    "Det recall":     lambda v: f"{v:.3f}",
    "Det precision":  lambda v: f"{v:.3f}",
    "Compliance acc": lambda v: f"{v:.3f}",
    "Helmet acc":     lambda v: f"{v:.3f}",
    "Vest acc":       lambda v: f"{v:.3f}",
}
print("\n" + "=" * 100)
print("End-to-End pipeline evaluation (compliance scored on matched workers)")
print("=" * 100)
print(summary.to_string(index=False, formatters=fmt))

summary.to_csv(f"{OUT_DIR}/summary.csv", index=False)
print(f"\nSaved: {OUT_DIR}/summary.csv")


In [ ]:
# =============================================================================
# Qualitative output: annotated sample images (green = compliant, red = not)
# =============================================================================
def annotate(img_bgr, workers):
    """Draw worker boxes coloured by compliance, with a short label."""
    out = img_bgr.copy()
    for w in workers:
        x1, y1, x2, y2 = w["box"]
        color = (0, 170, 0) if w["compliant"] else (0, 0, 220)   # BGR
        label = "OK" if w["compliant"] else "no " + "+".join(w["missing"])
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        ytxt = max(0, y1 - 6)
        cv2.putText(out, label, (x1, ytxt), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, color, 2, cv2.LINE_AA)
    return out


def sample_grid(pairs, title, out_path, n=6, seed=SEED):
    """Pick n images that have at least one detected worker and draw them."""
    rng = random.Random(seed)
    shuffled = pairs[:]
    rng.shuffle(shuffled)

    picked = []
    for img_path, _ in shuffled:
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            continue
        workers = run_pipeline(img_bgr)
        if workers:
            picked.append((annotate(img_bgr, workers), len(workers)))
        if len(picked) == n:
            break

    rows = (len(picked) + 2) // 3
    fig, axes = plt.subplots(rows, 3, figsize=(15, 5 * rows))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, (img, k) in zip(axes, picked):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{k} worker(s)", fontsize=10)
    plt.suptitle(title, fontsize=13, y=1.0)
    plt.tight_layout()
    plt.savefig(out_path, dpi=110, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")


sample_grid(in_domain_pairs, "End-to-End - Test In-Domain (Kaggle CSS test)",
            f"{OUT_DIR}/samples_test_in_domain.png")
sample_grid(ood_pairs, "End-to-End - Test Out-of-Domain (Ultralytics)",
            f"{OUT_DIR}/samples_test_out_of_domain.png")


In [ ]:
# =============================================================================
# Compliance confusion matrices on matched workers (both splits)
# =============================================================================
def plot_cm(ax, cm, title):
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["non-compliant", "compliant"])
    ax.set_yticklabels(["non-compliant", "compliant"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
    ax.set_title(title)
    total = cm.sum()
    for g in range(2):
        for p in range(2):
            pct = 100 * cm[g, p] / total if total else 0
            ax.text(p, g, f"{cm[g, p]}\n({pct:.1f}%)", ha="center", va="center",
                    color="white" if cm[g, p] > cm.max() / 2 else "black", fontsize=11)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_cm(axes[0], in_cm,  "Test In-Domain (matched workers)")
plot_cm(axes[1], ood_cm, "Test Out-of-Domain (matched workers)")
plt.suptitle("End-to-End compliance confusion - GT vs predicted", fontsize=12, y=1.03)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_DIR}/confusion_matrices.png")


## Next steps

All artifacts are now under `MyDrive/PPE_Project/end_to_end/`:

- `summary.csv` — the headline end-to-end table
- `samples_test_in_domain.png`, `samples_test_out_of_domain.png` — annotated examples
- `confusion_matrices.png` — compliance confusion on matched workers

**To finish Chapter 14:**
1. Sync these small files to the laptop and commit them under `results/pipeline/`.
2. Back in `01_main_pipeline.ipynb`, build Chapter 14 (display + findings) from `summary.csv`.
